In [1]:
import numpy as np
from tqdm import tqdm
import networkx as nx
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.applications import Maxcut
from qiskit_algorithms import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import NELDER_MEAD, COBYLA, L_BFGS_B
from qiskit_optimization.algorithms import MinimumEigenOptimizer, GurobiOptimizer
from qiskit_aer import AerSimulator, StatevectorSimulator
from qiskit.primitives import (
    BackendSampler,
    BackendSamplerV2,
    BackendEstimator,
    BackendEstimatorV2,
    StatevectorEstimator,
    Estimator,
)
from qiskit_aer.primitives import (
    Sampler as AerSampler,
    SamplerV2 as AerSamplerV2,
    Estimator as AerEstimator,
    EstimatorV2 as AerEstimatorV2,
)
from lcc import LCCVQE, FullVQE

# Generate Max-Cut instances

In [2]:
n = 10
# G = nx.random_regular_graph(3, n, seed=0)
G = nx.gnp_random_graph(n, 0.5, seed=0)
max_cut = Maxcut(G)
qp = max_cut.to_quadratic_program()

# Exact solution

In [3]:
exact_solver = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result = exact_solver.solve(qp)
true_obj = exact_result.fval
print(exact_result.prettyprint())

objective function value: 14.0
variable values: x_0=1.0, x_1=0.0, x_2=0.0, x_3=0.0, x_4=0.0, x_5=1.0, x_6=1.0, x_7=1.0, x_8=0.0, x_9=0.0
status: SUCCESS


In [4]:
gurobi_result = GurobiOptimizer().solve(qp)
true_obj = gurobi_result.fval
print('Gurobi cost:', true_obj)

Restricted license - for non-production use only - expires 2025-11-24
Gurobi cost: 14.0


# Solve using LCC

In [ ]:
estimator = StatevectorEstimator()
# estimator = None
optimizer = L_BFGS_B()
# optimizer = COBYLA(maxiter=200000)

n_trials = 2
shots = int(1e5)
costs = []
solutions = []

for _ in tqdm(range(n_trials)):
    solver = LCCVQE(
        qp,
        shots=shots,
        estimator=estimator,
        optimizer=optimizer,
    )

    result = solver.solve(initial_point=None, run_sampler=False)
    expectation = -(result.fun + solver.offset)
    costs.append(expectation)

    print('Expectation:', expectation)

 50%|█████     | 1/2 [00:27<00:27, 27.40s/it]

Expectation: 13.999999968806335


100%|██████████| 2/2 [00:55<00:00, 27.65s/it]

Expectation: 12.99999881737083


# Solve without using LCC

In [8]:
estimator = StatevectorEstimator()
# estimator = None
optimizer = L_BFGS_B()

n_trials = 2
shots = int(1e5)
costs = []
solutions = []

for _ in tqdm(range(n_trials)):
    solver = FullVQE(
        qp,
        shots=shots,
        estimator=estimator,
        optimizer=optimizer,
    )

    result = solver.solve(initial_point=None, run_sampler=False)
    expectation = -(result.fun + solver.offset)
    costs.append(expectation)

    print('Expectation:', expectation)

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:03<00:03,  3.19s/it]

Expectation: 12.999999999814019


100%|██████████| 2/2 [00:05<00:00,  2.60s/it]

Expectation: 12.999999999861544
